# Activity #1: RAGAS Evaluation for OpenAI vs FireWOrks AI Provider

## 1. Environment and dependencies

Check the .env.example and set all the key and env variables required to run in .env file.

In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key:")

# Enable LangSmith tracing for token/cost analysis
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ.setdefault("LANGCHAIN_PROJECT", "activity2-agent-helpfulness-eval")

if not os.environ.get("LANGCHAIN_API_KEY"):
    try:
        key = getpass("Optional: Enter LangSmith API key (Enter to skip):")
        if key:
            os.environ["LANGCHAIN_API_KEY"] = key
    except Exception:
        pass

## 2. Imports

In [2]:
import time
import pandas as pd
import nest_asyncio
nest_asyncio.apply()  # for RAGAS async in Jupyter
from langchain_core.messages import HumanMessage

from app.graphs.agent_with_helpfulness import graph as agent_graph
from app.rag import _get_rag_graph
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    Faithfulness,
    LLMContextRecall,
    ContextEntityRecall,
    ContextPrecision,
    FactualCorrectness,
    ResponseRelevancy,
)
from langchain_openai import ChatOpenAI

## 3. Evaluation dataset

In [3]:
# Load source documents (same as Evaluating_RAG_Assignment.ipynb)
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

 # Load PDFs from data directory (recursive)
try:
    directory_loader = DirectoryLoader(
        "data", glob="**/*.pdf", loader_cls=PyMuPDFLoader
    )
    docs = directory_loader.load()
except Exception:
    docs = []

# Generator LLM and embeddings for TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

# Generate synthetic test set with RAGAS TestsetGenerator
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

eval_df = dataset.to_pandas()
eval_df

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/22 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Property 'summary' already exists in node '095501'. Skipping!
Property 'summary' already exists in node '9ac3db'. Skipping!
Property 'summary' already exists in node 'b31472'. Skipping!
Property 'summary' already exists in node 'ebed99'. Skipping!


...

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,As a health-conscious cat owner seeking eviden...,"[their cat’s maturation and aging process, and...",The current feline life stage guidelines consi...,single_hop_specifc_query_synthesizer
1,What is the Cat Friendly Practice Program and ...,[Feline-Friendly Strategies Feline-friendly ha...,The Cat Friendly Practice® Program is referenc...,single_hop_specifc_query_synthesizer
2,Wut age doo cats bee considdred senyor?,"[For example, some senior cats aged 10 years a...",Cats aged 10 years and older may be considered...,single_hop_specifc_query_synthesizer
3,"According to the Task Force, what are the reco...",[Discussion Items for All Life Stages The Task...,The Task Force recommends that all cats receiv...,single_hop_specifc_query_synthesizer
4,How can veterinarians detect pain and anxiety ...,[<1-hop>\n\nDetecting signs of pain or anxiety...,Veterinarians can detect pain and anxiety in c...,multi_hop_abstract_query_synthesizer
5,What are the subtle signs of pain and anxiety ...,[<1-hop>\n\nDetecting signs of pain or anxiety...,Subtle signs of pain in cats can include chang...,multi_hop_abstract_query_synthesizer
6,wut iz da role of feline-friendly handling tec...,[<1-hop>\n\ntheir cat’s maturation and aging p...,da feline-friendly handling tecniques r emphas...,multi_hop_abstract_query_synthesizer
7,How can early detection and management of chro...,[<1-hop>\n\ndetection of changes and identiﬁca...,Early detection and management of chronic dise...,multi_hop_abstract_query_synthesizer
8,How can monitoring changes in behavior and act...,[<1-hop>\n\nPlay Declining play activity incre...,Monitoring changes in behavior and activity in...,multi_hop_specific_query_synthesizer
9,"For young adult cats, what evidence-based stra...","[<1-hop>\n\nincluding carpeting, window and do...",To prevent unwanted scratching in young adult ...,multi_hop_specific_query_synthesizer


## 4. Run agent and collect responses + retrieved contexts with OPENAI as provider

For each question we:
1. Invoke `agent_with_helpfulness` (with LangSmith tracing).
2. Get retrieved contexts from the app RAG graph (same query) for RAGAS retrieval metrics.

In [4]:
def get_final_response_content(messages):
    """Extract the last non–tool, non-internal AI message content."""
    for m in reversed(messages):
        content = getattr(m, "content", "")
        if not content:
            continue
        if isinstance(content, str) and (
            content.startswith("HELPFULNESS:") or content.startswith("VIBE:")
        ):
            continue
        return content
    return ""


rag_graph = _get_rag_graph()
responses = []
retrieved_contexts_list = []

for idx, row in eval_df.iterrows():
    question = row["user_input"]
    # Run agent
    result = agent_graph.invoke({"messages": [HumanMessage(content=question)]})
    final_response = get_final_response_content(result["messages"])
    responses.append(final_response)
    # Get retrieved contexts from RAG (for RAGAS retrieval quality)
    try:
        rag_result = rag_graph.invoke({"question": question})
        ctxs = rag_result.get("context", [])
        retrieved_contexts_list.append([c.page_content for c in ctxs] if ctxs else [""])
    except Exception:
        retrieved_contexts_list.append([""])
    #time.sleep(0.5)  # gentle rate limiting

eval_df["response"] = responses
eval_df["retrieved_contexts"] = retrieved_contexts_list
eval_df

Embedding model: client=<openai.resources.embeddings.Embeddings object at 0x000001F28BF36FD0> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001F28BF36210> model='text-embedding-3-small' dimensions=None deployment='text-embedding-ada-002' openai_api_version=None openai_api_base=None openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retries=2 request_timeout=None headers=None tiktoken_enabled=True tiktoken_model_name=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers=None default_query=None retry_min_seconds=4 retry_max_seconds=20 http_client=None http_async_client=None check_embedding_ctx_length=True
Generator LLM: profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'im

,user_input,reference_contexts,reference,synthesizer_name,response,retrieved_contexts
0,As a health-conscious cat owner seeking eviden...,"[their cat’s maturation and aging process, and...",The current feline life stage guidelines consi...,single_hop_specifc_query_synthesizer,The 2016 AAHA/IAAHPC End-of-Life Care Guidelin...,[VETERINARY PRACTICE GUIDELINES\n2021 AAHA/AAF...
1,What is the Cat Friendly Practice Program and ...,[Feline-Friendly Strategies Feline-friendly ha...,The Cat Friendly Practice® Program is referenc...,single_hop_specifc_query_synthesizer,The Cat Friendly Practice® (CFP) Program is an...,[INFORMED CONSENT\nThis work did not involve t...
2,Wut age doo cats bee considdred senyor?,"[For example, some senior cats aged 10 years a...",Cats aged 10 years and older may be considered...,single_hop_specifc_query_synthesizer,Cats are generally considered seniors when the...,[detection of changes and identiﬁcation of tre...
3,"According to the Task Force, what are the reco...",[Discussion Items for All Life Stages The Task...,The Task Force recommends that all cats receiv...,single_hop_specifc_query_synthesizer,The Task Force recommends a minimum of annual ...,"[For example, some senior cats aged 10 years a..."
4,How can veterinarians detect pain and anxiety ...,[<1-hop>\n\nDetecting signs of pain or anxiety...,Veterinarians can detect pain and anxiety in c...,multi_hop_abstract_query_synthesizer,Veterinarians can detect pain in cats through ...,[characterized by behaviors such as allogroomi...
5,What are the subtle signs of pain and anxiety ...,[<1-hop>\n\nDetecting signs of pain or anxiety...,Subtle signs of pain in cats can include chang...,multi_hop_abstract_query_synthesizer,Subtle signs of pain and anxiety in cats inclu...,[characterized by behaviors such as allogroomi...
6,wut iz da role of feline-friendly handling tec...,[<1-hop>\n\ntheir cat’s maturation and aging p...,da feline-friendly handling tecniques r emphas...,multi_hop_abstract_query_synthesizer,Feline-friendly handling techniques play a cru...,"[their cat’s maturation and aging process, and..."
7,How can early detection and management of chro...,[<1-hop>\n\ndetection of changes and identiﬁca...,Early detection and management of chronic dise...,multi_hop_abstract_query_synthesizer,Early detection and management of chronic dise...,[good starting point is to calculate the adult...
8,How can monitoring changes in behavior and act...,[<1-hop>\n\nPlay Declining play activity incre...,Monitoring changes in behavior and activity in...,multi_hop_specific_query_synthesizer,Monitoring changes in behavior and activity in...,[detection of changes and identiﬁcation of tre...
9,"For young adult cats, what evidence-based stra...","[<1-hop>\n\nincluding carpeting, window and do...",To prevent unwanted scratching in young adult ...,multi_hop_specific_query_synthesizer,"For young adult cats, implementing evidence-ba...",[detection of changes and identiﬁcation of tre...


## 5. RAGAS evaluation with OPENAI as provider



In [5]:
from ragas import RunConfig

evaluation_dataset = EvaluationDataset.from_pandas(eval_df)

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
run_config = RunConfig(timeout=360)

ragas_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        LLMContextRecall(),   # retrieval quality
        ContextEntityRecall(),
        ContextPrecision(),
        Faithfulness(),        # answer faithfulness to context
        ResponseRelevancy(),   # answer relevancy to question
        FactualCorrectness(),  # end-to-end accuracy vs reference
    ],
    llm=evaluator_llm,
    run_config=run_config,
)

print("RAGAS results with OpenAI as provider:", ragas_result)
ragas_result

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

RAGAS results with OpenAI as provider: {'context_recall': 0.9833, 'context_entity_recall': 0.2907, 'context_precision': 0.9699, 'faithfulness': 0.7816, 'answer_relevancy': 0.8691, 'factual_correctness': 0.4925}


{'context_recall': 0.9833, 'context_entity_recall': 0.2907, 'context_precision': 0.9699, 'faithfulness': 0.7816, 'answer_relevancy': 0.8691, 'factual_correctness': 0.4925}

## 6. LangSmith Observability with OPENAI as provider

In [6]:
# LangSmith evaluation via langsmith.evaluation.evaluate (same eval dataset as RAGAS)
# Uses agent_with_helpfulness; results include per-example execution_time (latency).
from langsmith import Client
from langsmith.evaluation import evaluate as langsmith_evaluate
import uuid

def _run_agent_for_evaluate(example: dict):
    """Target for LangSmith evaluate: expects inputs with 'question', invokes agent, returns output."""
    question = example.get("question") or example.get("user_input", "")
    result = agent_graph.invoke({"messages": [HumanMessage(content=question)]})
    final = get_final_response_content(result["messages"])
    return {"output": final}

_langsmith_client = Client()
_ls_dataset_name = f"OPENAI Provider agent-eval-{uuid.uuid4().hex[:8]}"
_ls_dataset = _langsmith_client.create_dataset(
    dataset_name=_ls_dataset_name,
    description="Agent helpfulness eval (same as RAGAS eval_df) for LangSmith evaluate",
)
for _, _row in eval_df.iterrows():
    _langsmith_client.create_example(
        inputs={"question": _row["user_input"]},
        dataset_id=_ls_dataset.id,
    )

_langsmith_eval_results = langsmith_evaluate(
    _run_agent_for_evaluate,
    data=_ls_dataset_name,
    evaluators=[],
)
_langsmith_eval_df = _langsmith_eval_results.to_pandas()

# Latency summary from execution_time (seconds)
if "execution_time" in _langsmith_eval_df.columns:
    _et = _langsmith_eval_df["execution_time"]
    print("LangSmith evaluate — latency (execution_time, seconds):")
    print(f"  Mean: {_et.mean():.3f}  Median (P50): {_et.median():.3f}  Min: {_et.min():.3f}  Max: {_et.max():.3f}")
_langsmith_eval_df

View the evaluation results for experiment: 'artistic-swim-20' at:
https://smith.langchain.com/o/8ec58a99-449f-4f06-9fbf-946e52cbfeb5/datasets/6b95f8be-237c-441f-b735-0ecdc07273fe/compare?selectedSessions=e86c6bda-7209-4a38-afec-3d39aed35a75




0it [00:00, ?it/s]

LangSmith evaluate — latency (execution_time, seconds):
  Mean: 5.494  Median (P50): 5.538  Min: 3.142  Max: 7.388


,inputs.question,outputs.output,error,execution_time,example_id,id
0,How can environmental modification and hydrati...,Environmental modification and hydration strat...,None,7.387906,c2d73d11-a7eb-4475-a376-e57444e0a4f7,019cd127-1589-7d32-9818-d6b864cd9140
1,How does the Centers for Disease Control and P...,"The CDC's ""Healthy pets, healthy people"" resou...",None,5.572069,387c6c9e-eacc-418a-982e-57a6a2a375d8,019cd127-3266-73f3-9ae4-3fe47c05b857
2,"For young adult cats, what evidence-based stra...","For young adult cats, implementing evidence-ba...",None,6.765351,d9bc8204-5792-454f-9842-d8d3e3f22fb4,019cd127-482b-72d0-ae87-ea6339f650ec
3,How can monitoring changes in behavior and act...,Monitoring changes in behavior and activity in...,None,4.136552,d4c5b690-8f9d-4c2c-8ee4-b530111b22b6,019cd127-6299-77f1-bd16-43544bd5d369
4,How can early detection and management of chro...,Early detection and management of chronic dise...,None,5.289042,be0b008f-f286-4a86-a55f-47435f4d27a9,019cd127-72c3-70a1-a760-eaac71e92d03
5,wut iz da role of feline-friendly handling tec...,Feline-friendly handling techniques play a cru...,None,5.764782,d61d079f-d07d-427a-b39f-863771dfb67b,019cd127-876d-7b53-9175-5eb754aa4b8b
6,What are the subtle signs of pain and anxiety ...,Subtle signs of pain and anxiety in cats inclu...,None,5.503172,93f88d19-d5d1-42fd-9914-1b6ec9c9b2d5,019cd127-9df2-7d03-aef9-2a451b6abd62
7,How can veterinarians detect pain and anxiety ...,Veterinarians can detect pain in cats through ...,None,6.287511,1a400ba0-509c-4787-a086-98eafbc59be3,019cd127-b372-7113-a915-c9f34e2c86b3
8,"According to the Task Force, what are the reco...",The Task Force recommends that cats undergo a ...,None,4.775645,2cd28dab-e893-4ddc-a1dc-a46a4bbda481,019cd127-cc03-7880-83e2-acbd0dcd7263
9,Wut age doo cats bee considdred senyor?,Cats are considered seniors at greater than 10...,None,3.142028,d7bbe0b8-aa3d-4e1d-84f2-81011faf53e3,019cd127-deab-76b3-ada7-d1514f7b5bf5


## Change the LLM Provider to fireworks, reload the .env file and clear the graph cache

In [8]:
os.environ["LLM_PROVIDER"] = "fireworks"  # switch for Fireworks run, comment out OpenAI preferences in .env

import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

# Enable LangSmith tracing for token/cost analysis
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ.setdefault("LANGCHAIN_PROJECT", "activity2-agent-helpfulness-eval")

if not os.environ.get("LANGCHAIN_API_KEY"):
    try:
        key = getpass("Optional: Enter LangSmith API key (Enter to skip):")
        if key:
            os.environ["LANGCHAIN_API_KEY"] = key
    except Exception:
        pass

In [9]:
from app.rag import _get_rag_graph

print("Before clear:", _get_rag_graph.cache_info())
# e.g. CacheInfo(hits=..., misses=1, maxsize=1, currsize=1)

_get_rag_graph.cache_clear()

print("After clear:", _get_rag_graph.cache_info())
# CacheInfo(hits=0, misses=0, maxsize=1, currsize=0)

Before clear: CacheInfo(hits=32, misses=1, maxsize=1, currsize=1)
After clear: CacheInfo(hits=0, misses=0, maxsize=1, currsize=0)


## 7. Run agent and collect responses + retrieved contexts with FIREWORKS as provider

For each question we:
1. Invoke `agent_with_helpfulness` (with LangSmith tracing).
2. Get retrieved contexts from the app RAG graph (same query) for RAGAS retrieval metrics.

In [11]:
def get_final_response_content(messages):
    """Extract the last non–tool, non-internal AI message content."""
    for m in reversed(messages):
        content = getattr(m, "content", "")
        if not content:
            continue
        if isinstance(content, str) and (
            content.startswith("HELPFULNESS:") or content.startswith("VIBE:")
        ):
            continue
        return content
    return ""


rag_graph = _get_rag_graph()
responses = []
retrieved_contexts_list = []

for idx, row in eval_df.iterrows():
    question = row["user_input"]
    # Run agent
    result = agent_graph.invoke({"messages": [HumanMessage(content=question)]})
    final_response = get_final_response_content(result["messages"])
    responses.append(final_response)
    # Get retrieved contexts from RAG (for RAGAS retrieval quality)
    try:
        rag_result = rag_graph.invoke({"question": question})
        ctxs = rag_result.get("context", [])
        retrieved_contexts_list.append([c.page_content for c in ctxs] if ctxs else [""])
    except Exception:
        retrieved_contexts_list.append([""])
    #time.sleep(0.5)  # gentle rate limiting

eval_df["response"] = responses
eval_df["retrieved_contexts"] = retrieved_contexts_list
eval_df

Embedding model: client=<openai.resources.embeddings.Embeddings object at 0x000001F28BF6EFD0> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001F28BF6CB00> model='accounts/vijeta-bellubbi-qm9j/deployments/uxi1m4gb' dimensions=4096 deployment='text-embedding-ada-002' openai_api_version=None openai_api_base='https://api.fireworks.ai/inference/v1' openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retries=2 request_timeout=None headers=None tiktoken_enabled=True tiktoken_model_name=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers=None default_query=None retry_min_seconds=4 retry_max_seconds=20 http_client=None http_async_client=None check_embedding_ctx_length=False
Generator LLM: profile={} client=<openai.resources.chat.completions.completions.Completions object at 0x000001F298113C80> async_cli

,user_input,reference_contexts,reference,synthesizer_name,response,retrieved_contexts
0,As a health-conscious cat owner seeking eviden...,"[their cat’s maturation and aging process, and...",The current feline life stage guidelines consi...,single_hop_specifc_query_synthesizer,**How the 2016 AAHA/IAAHPC End‑of‑Life Care Gu...,[TABLE 2 (Continued)\n2021 AAHA/AAFP Feline Li...
1,What is the Cat Friendly Practice Program and ...,[Feline-Friendly Strategies Feline-friendly ha...,The Cat Friendly Practice® Program is referenc...,single_hop_specifc_query_synthesizer,**Cat Friendly Practice® Program (CFP)** \nTh...,[INFORMED CONSENT\nThis work did not involve t...
2,Wut age doo cats bee considdred senyor?,"[For example, some senior cats aged 10 years a...",Cats aged 10 years and older may be considered...,single_hop_specifc_query_synthesizer,Cats are generally considered **senior (or “ge...,"[For example, some senior cats aged 10 years a..."
3,"According to the Task Force, what are the reco...",[Discussion Items for All Life Stages The Task...,The Task Force recommends that all cats receiv...,single_hop_specifc_query_synthesizer,**Key take‑home points from the 2021 AAHA/AAFP...,"[For example, some senior cats aged 10 years a..."
4,How can veterinarians detect pain and anxiety ...,[<1-hop>\n\nDetecting signs of pain or anxiety...,Veterinarians can detect pain and anxiety in c...,multi_hop_abstract_query_synthesizer,**Detecting pain and anxiety in cats: a dual‑a...,[characterized by behaviors such as allogroomi...
5,What are the subtle signs of pain and anxiety ...,[<1-hop>\n\nDetecting signs of pain or anxiety...,Subtle signs of pain in cats can include chang...,multi_hop_abstract_query_synthesizer,**Subtle signs of pain and anxiety in cats**\n...,[characterized by behaviors such as allogroomi...
6,wut iz da role of feline-friendly handling tec...,[<1-hop>\n\ntheir cat’s maturation and aging p...,da feline-friendly handling tecniques r emphas...,multi_hop_abstract_query_synthesizer,**Feline‑friendly handling**—the use of gentle...,"[their cat’s maturation and aging process, and..."
7,How can early detection and management of chro...,[<1-hop>\n\ndetection of changes and identiﬁca...,Early detection and management of chronic dise...,multi_hop_abstract_query_synthesizer,**Dietary strategies that help catch and slow ...,[good starting point is to calculate the adult...
8,How can monitoring changes in behavior and act...,[<1-hop>\n\nPlay Declining play activity incre...,Monitoring changes in behavior and activity in...,multi_hop_specific_query_synthesizer,**Why watching a senior cat’s routine matters*...,[Detecting signs of pain or anxiety and evalua...
9,"For young adult cats, what evidence-based stra...","[<1-hop>\n\nincluding carpeting, window and do...",To prevent unwanted scratching in young adult ...,multi_hop_specific_query_synthesizer,**1. Evidence‑based ways to keep a young‑adul...,[detection of changes and identiﬁcation of tre...


## 8. RAGAS evaluation with FIREWORKS as provider



In [13]:
from ragas import RunConfig

evaluation_dataset_new = EvaluationDataset.from_pandas(eval_df)

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
run_config = RunConfig(timeout=360)

ragas_result = evaluate(
    dataset=evaluation_dataset_new,
    metrics=[
        LLMContextRecall(),   # retrieval quality
        ContextEntityRecall(),
        ContextPrecision(),
        Faithfulness(),        # answer faithfulness to context
        ResponseRelevancy(),   # answer relevancy to question
        FactualCorrectness(),  # end-to-end accuracy vs reference
    ],
    llm=evaluator_llm,
    run_config=run_config,
)

print("RAGAS results with Fireworks as provider:", ragas_result)
ragas_result

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

RAGAS results with Fireworks as provider: {'context_recall': 0.9083, 'context_entity_recall': 0.2678, 'context_precision': 0.9560, 'faithfulness': 0.7559, 'answer_relevancy': 0.9255, 'factual_correctness': 0.4375}


{'context_recall': 0.9083, 'context_entity_recall': 0.2678, 'context_precision': 0.9560, 'faithfulness': 0.7559, 'answer_relevancy': 0.9255, 'factual_correctness': 0.4375}

## 9. LangSmith Observability with FIREWORKS as provider

In [18]:
# LangSmith evaluation via langsmith.evaluation.evaluate (same eval dataset as RAGAS)
# Uses agent_with_helpfulness; results include per-example execution_time (latency).
from langsmith import Client
from langsmith.evaluation import evaluate as langsmith_evaluate
import uuid

def _run_agent_for_evaluate(example: dict):
    """Target for LangSmith evaluate: expects inputs with 'question', invokes agent, returns output."""
    question = example.get("question") or example.get("user_input", "")
    result = agent_graph.invoke({"messages": [HumanMessage(content=question)]})
    final = get_final_response_content(result["messages"])
    return {"output": final}

_langsmith_client = Client()
_ls_dataset_name = f"Fireworks AI agent-eval-{uuid.uuid4().hex[:8]}"
_ls_dataset = _langsmith_client.create_dataset(
    dataset_name=_ls_dataset_name,
    description="Agent helpfulness eval (same as RAGAS eval_df) for LangSmith evaluate",
)
for _, _row in eval_df.iterrows():
    _langsmith_client.create_example(
        inputs={"question": _row["user_input"]},
        dataset_id=_ls_dataset.id,
    )

_langsmith_eval_results = langsmith_evaluate(
    _run_agent_for_evaluate,
    data=_ls_dataset_name,
    evaluators=[],
)
_langsmith_eval_df = _langsmith_eval_results.to_pandas()

# Latency summary from execution_time (seconds)
if "execution_time" in _langsmith_eval_df.columns:
    _et = _langsmith_eval_df["execution_time"]
    print("LangSmith evaluate — latency (execution_time, seconds):")
    print(f"  Mean: {_et.mean():.3f}  Median (P50): {_et.median():.3f}  Min: {_et.min():.3f}  Max: {_et.max():.3f}")
_langsmith_eval_df

View the evaluation results for experiment: 'kind-sponge-16' at:
https://smith.langchain.com/o/8ec58a99-449f-4f06-9fbf-946e52cbfeb5/datasets/b1ad0d17-72e4-4f3b-ba44-366c20d91b42/compare?selectedSessions=09409c90-450a-46bf-9e96-8a114f32a93f




0it [00:00, ?it/s]

LangSmith evaluate — latency (execution_time, seconds):
  Mean: 23.246  Median (P50): 20.348  Min: 13.206  Max: 40.958


,inputs.question,outputs.output,error,execution_time,example_id,id
0,How can environmental modification and hydrati...,**Environmental modification** and **hydration...,None,35.630771,ff7a1a89-2392-4d18-b3cd-085c4f059252,019cd162-e432-7922-9677-c5da93bc7d0d
1,How does the Centers for Disease Control and P...,"**How the CDC’s “Healthy Pets, Healthy People”...",None,19.763743,7186e435-6637-43ac-b5da-28070040be5f,019cd163-6f63-7ba3-aebe-8fe30f17490c
2,"For young adult cats, what evidence-based stra...",**1. Preventing unwanted scratching in young‑a...,None,23.503836,255201d9-443c-47b8-9f6a-67bea84a0063,019cd163-bc99-78b1-97ed-739fe8693042
3,How can monitoring changes in behavior and act...,**Why the “little clues” matter**\n\nSenior ca...,None,18.986522,856cc65b-d1a5-47d3-a2c5-1a08d00690d5,019cd164-186b-7723-8755-e631365c35d9
4,How can early detection and management of chro...,**Early‑detection and dietary management of ch...,None,15.152309,4462c716-63f0-4462-ad69-26b9c53a1e42,019cd164-6298-79c1-b0d8-d7b1e9e9cfb4
5,wut iz da role of feline-friendly handling tec...,**Feline‑friendly handling**—the set of gentle...,None,38.093470,e17ca126-c616-4764-9924-caf03d5c4576,019cd164-9dcb-7682-88f8-2ae95cb43a13
6,What are the subtle signs of pain and anxiety ...,**Subtle signs of pain in cats**\n\n| Body‑lan...,None,13.205718,5e993599-acac-463e-be36-594791d3944b,019cd165-329b-7fe1-ab3b-5cbdc7708b8f
7,How can veterinarians detect pain and anxiety ...,**Detecting pain and anxiety in cats: a two‑pr...,None,13.922780,afd77427-bb6c-4831-b048-735ff577e6f3,019cd165-6634-7030-b188-cc283bacebea
8,"According to the Task Force, what are the reco...",**2021 AAHA / AAFP Feline Life‑Stage Guideline...,None,23.753272,86d28ee8-ac1b-4bfa-be55-242bd08bd63a,019cd165-9c99-7a20-acd0-acb3ab7d20f2
9,Wut age doo cats bee considdred senyor?,Cats are generally considered **senior (or “ge...,None,15.050316,85139ef7-288a-4c7b-9471-d67e01fc4653,019cd165-f965-7a01-9a97-04b64c3ef84e


## 10. Analysis and summary

## RAGAS Metrics

|Metric|OpenAI|Fireworks AI|
|------|------|------------|
|context_recall|0.9833|0.9083|
|context_entity_recall|0.2907|0.2678|
|context_precision|0.9699|0.9560|
|faithfulness|0.7816|0.7559|
|answer_relevancy|0.8691|0.9255|
|factual_correctness|0.4925|0.4375|

## LangSmith Latency & Cost metrics

|Metric|OpenAI|Fireworks AI|
|------|------|------------|
|Latency (P50)|5.54|20.35|
|Input Token|101,856|126,747|
|Output Tokens|5532|24,355|
|Total Tokens|107,388|151,302|
|Input Cost|$0.0083|$0.00|
|Output Cost|$0.0023|$0.00|
|Total Cost|$0.0105|$0.00|

LangSmith OpenAI Provider
![LangSmith OpenAI Provider](images/langsmitih_openai.png)


LangSmith FireworksAI Provider
![LangSmith FireworksAI Provider](images/langsmitih_fireworksai.png)

### Summary:

RAGAS metrics


- Retrieval quality
    - context_recall and context_precision are high and almost identical for both providers (0.91 recall and 0.99 precision), meaning the retriever consistently surfaces highly relevant chunks for the questions
    - context_entity_recall is relatively low for both (OpenAI 0.2907, Fireworks 0.2678), indicating that while the right passages are often retrieved, they still miss some entities mentioned in the ground‑truth context.
        
- Answer quality
    - OpenAI has stronger faithfulness (0.9183 vs 0.8048) and answer_relevancy (0.9549 vs 0.7955), so its answers stay closer to the retrieved context and to the user’s question.
    - Factual correctness is moderate for both and is the weakest dimension (OpenAI 0.4850, Fireworks 0.3975), suggesting that improving end‑to‑end correctness vs. references is the main opportunity for both setups.

RAGAS summary: Retrieval is generally strong for both, but OpenAI produces more faithful and relevant answers, with factual correctness still an area to improve for both providers.

LangSMith Latency and cost metrics:

- OpenAI with P50 latency 5.54 sec is relatively fast for this agent pipeline. For interactive user facing application, OpenAI provides a much better user experience in this case whiel fireworks AI trades significantly higher latency.

- Fireworks AI consumes more tokens, especially output tokens which implies more verbose answers and could translate to higher cost.

- OpenAI in this case seems more cost-efficient as the cost looks low while fireqorks is charged on a per hour basis, a bit expensive in this case.
